# 05 · Residue-level interaction maps

Show interaction classes using residue identifiers from the structures. Absence in an event table means no event reported by the extraction helper, not proof that an interaction is biologically impossible.

## Configuration

Use `config.local.json` as described in [the workflow guide](../docs/WORKFLOW.md). Inputs must be authorized for local analysis. Outputs are cleared in version control; no research result is asserted by this notebook.

In [ ]:
from pathlib import Path
import sys
root = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'analysis/interactions.py').is_file()), None)
if root is None:
    raise FileNotFoundError('Start Jupyter from this repository')
sys.path.insert(0, str(root))
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from analysis.notebook_setup import load_config, load_manifest
config = load_config()
out = config['output_dir']


## Prepare interaction matrix

In [ ]:
events = pd.read_csv(out / 'interaction_events.csv', dtype={'compound_id': str, 'pose_id': str})
if events.empty:
    raise ValueError('No interaction events to visualize')
events['residue_key'] = events.prot_chain.astype(str) + ':' + events.prot_res_num.astype(str)
matrix = events.assign(present=1).pivot_table(index=['compound_id', 'pose_id', 'feature'],
    columns='residue_key', values='present', aggfunc='max', fill_value=0)
matrix.to_csv(out / 'residue_interaction_presence.csv')

## Plot presence map

In [ ]:
import seaborn as sns
fig, ax = plt.subplots(figsize=(12, max(4, min(18, len(matrix) * .15))))
sns.heatmap(matrix, cmap=['#f5f8fb', '#246b9b'], cbar=False, ax=ax)
ax.set(title='Predicted residue-level interaction patterns', xlabel='Chain:residue', ylabel='Compound / pose / feature')
fig.tight_layout(); fig.savefig(out / 'residue_interaction_heatmap.svg')
plt.show()